# BLS Signature Scheme Usage

This notebook demonstrates the usage of the BLS (Boneh-Lynn-Shacham) signature scheme implemented in this project using the `py-ecc` library and the BLS12-381 curve.

## 1. Setup and Imports

First, we need to ensure the project modules are accessible. If you're running this from the `notebooks/` directory, we'll add the parent directory to the path.

In [ ]:
import os
import sys

module_path = os.path.abspath(os.path.join(".."))
if module_path not in sys.path:
    sys.path.append(module_path)

In [ ]:
from src.bls import BLS

## 2. Key Generation

BLS keys consist of a private key (a scalar) and a public key (a point on the G1 group of the BLS12-381 curve).

In [ ]:
sk, pk = BLS.key_gen()

print(f"Private Key: {sk}")
print(f"Public Key (hex): {pk.hex()}")
print(f"Public Key Length: {len(pk)} bytes")

## 3. Signing and Verification

The signature is a point on the G2 group. One of the main advantages of BLS is its short signature size.

In [ ]:
message = b"Cryptography research is fascinating!"
signature = BLS.sign(sk, message)

print(f"Message: {message.decode()}")
print(f"Signature (hex): {signature.hex()}")
print(f"Signature Length: {len(signature)} bytes")

# Verification
is_valid = BLS.verify(pk, message, signature)
print(f"\nIs the signature valid? {is_valid}")

## 4. Testing Invalid Scenarios

Let's verify that the scheme correctly identifies invalid signatures or tampered messages.

In [ ]:
tampered_message = b"Cryptography research is BORING!"
is_valid_tampered = BLS.verify(pk, tampered_message, signature)
print(f"Verification with tampered message: {is_valid_tampered}")

_, other_pk = BLS.key_gen()
is_valid_other_pk = BLS.verify(other_pk, message, signature)
print(f"Verification with wrong public key: {is_valid_other_pk}")

## 5. Signature Aggregation

One of the most powerful features of BLS is the ability to aggregate multiple signatures from different signers (on the same message) into a single signature.

In [ ]:
n_signers = 5
common_message = b"Collective agreement on the block state."

# Each signer generates their own keypair and signs the same message
signers = [BLS.key_gen() for _ in range(n_signers)]
signatures = [BLS.sign(sk, common_message) for sk, pk in signers]
public_keys = [pk for sk, pk in signers]

# Aggregate all signatures into one
aggregated_sig = BLS.aggregate_signatures(signatures)

print(f"Aggregated Sig Length: {len(aggregated_sig)} bytes")

# Verify the aggregated signature against all public keys
is_agg_valid = BLS.verify_aggregated(public_keys, common_message, aggregated_sig)
print(f"Is the aggregated signature valid? {is_agg_valid}")